## Dataset - Porto Seguro Safe Driver Prediction

In [ ]:
df_train = pd.read_csv('../Cleaned-Dataset/train.csv')
df_test = pd.read_csv('../Cleaned-Dataset/test.csv')


In [ ]:
train_samples = int(df_train.shape[0] * 0.8)

X_train, y_train = df_train[:train_samples].drop(columns=['target']), df_train[:train_samples]['target']
X_valid, y_valid = df_train[train_samples:].drop(columns=['target']), df_train[train_samples:]['target']

test = df_test

In [ ]:
# Encoding 
# Target encode only high-cardinality column
cat_cols_target = ["ps_car_11_cat"]

cat_cols_final = [col for col in X_train.columns if col.endswith("_cat")]
cat_cols_onehot = [col for col in cat_cols_final if col not in cat_cols_target]

from category_encoders import TargetEncoder, OneHotEncoder
# --- Target Encoding ---
target_encoder = TargetEncoder(cols=cat_cols_target, smoothing=0.3)

X_train = target_encoder.fit_transform(X_train, y_train)
X_valid = target_encoder.transform(X_valid)
test = target_encoder.transform(test)

# --- One Hot Encoding ---
onehot_encoder = OneHotEncoder(cols=cat_cols_onehot, use_cat_names=True)

X_train = onehot_encoder.fit_transform(X_train)
X_valid = onehot_encoder.transform(X_valid)
test = onehot_encoder.transform(test)

## Decision Tree XGBoost Style


In [ ]:
import numpy as np
import pandas as pd

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, leaf_weight=None, leaf_count=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.leaf_count = leaf_count
        self.leaf_weight = leaf_weight

class DecisionTreeXGBoostStyle:
    """Decision Tree while focusing on time optimization and as weak learner."""
    def __init__(self, 
                max_depth=10,
                min_child_weight=1.0, 
                colsample_bytree=0.8, 
                reg_alpha=0.0,
                reg_gamma=0.0,
                reg_lambda=1.0,
                subsample=0.8
        ):
        self.max_depth = max_depth
        self.min_child_weight = min_child_weight
        self.colsample_bytree  = colsample_bytree
        self.reg_alpha = reg_alpha
        self.reg_gamma = reg_gamma
        self.reg_lambda = reg_lambda
        self.subsample = subsample
        self.root = None
    
    def _best_split(self, X, grad, hess, indices):
        m = len(indices)

        if m < self.min_samples_split:
            return None, None, None

        best_gain = float('inf')
        best_feature, best_threshold = None, None

        n_features = X.shape[1]

        # Column Sampling 
        if self.colsample_bytree is None: 
            colsample = 1.0 
        else: 
            colsample = self.colsample_bytree 
            
        if colsample < 1.0: 
            n_select = max(1, int(colsample * n_features)) 
            features = np.random.choice(n_features, n_select, replace=False) 
        else: 
            features = np.arange(n_features)
        
        # Precompute total G and H 
        G = np.sum(grad[indices]) 
        H = np.sum(hess[indices]) 
        
        g = grad[indices] 
        h = hess[indices]

        for feature in features:
            X_col = X[indices, feature]

            # Sort once 
            order = np.argsort(X_col, kind='mergesort') 
            X_sorted = X_col[order] 
            g_sorted = g[order] 
            h_sorted = h[order] 
            
            # Cumulative sums (vectorized) 
            G_L = np.cumsum(g_sorted)[:-1] 
            H_L = np.cumsum(h_sorted)[:-1] 
            G_R = G - G_L 
            H_R = H - H_L

            # Avoid duplicate feature values 
            diff = X_sorted[1:] != X_sorted[:-1] 
            
            # Valid splits (min_child_weight constraint) 
            valid = (H_L >= self.min_child_weight) & (H_R >= self.min_child_weight) 
            
            if not np.any(valid): 
                continue 
            
            valid = diff & valid 
            
            if not np.any(valid): 
                continue

            # Vectorized gain calculation 
            eps = 1e-12

            left_gain = (G_L**2) / (H_L + self.reg_lambda + eps) 
            right_gain = (G_R**2) / (H_R + self.reg_lambda + eps) 
            parent_gain = (G**2) / (H + self.reg_lambda + eps) 

            gain = 0.5 * (left_gain + right_gain - parent_gain) - self.reg_gamma 
            
            gain[~valid] = -np.inf 
            
            idx = np.argmax(gain) 
            
            if gain[idx] > best_gain: 
                best_gain = gain[idx] 
                best_feature = feature 
                best_threshold = (X_sorted[idx] + X_sorted[idx + 1]) / 2.0

            if best_gain <= self.reg_gamma:
                return None, None, None

        return best_feature, best_threshold, best_gain

    
    # Recursive tree building (No Array Copying)
    def _build_tree(self, X, grad, hess, indices, depth=0):
        """Recursive tree building."""
        m = len(indices)
        G = np.sum(grad[indices])
        H = np.sum(hess[indices])

        # stopping condition.
        if ((self.max_depth is not None and depth >= self.max_depth) or 
            H < self.self.min_child_weight or 
            m < 2):
            # Optimal leaf weight with L1 + L2
            if self.reg_alpha > 0:
                w = -np.sign(G) * max(0, abs(G) - self.reg_alpha) / (H + self.reg_lambda)
            else:
                w = - G/(H + self.reg_lambda)
            return Node(leaf_weight=w, leaf_count=m)
        
        feature, threshold = self._best_split(X, y, indices)
        if feature is None: 
            # Optimal leaf weight with L1 + L2
            if self.reg_alpha > 0:
                w = -np.sign(G) * max(0, abs(G) - self.reg_alpha) / (H + self.reg_lambda)
            else:
                w = - G/(H + self.reg_lambda)
            return Node(leaf_weight=w, leaf_count=m)
        
        mask = X[indices, feature] <= threshold

        left_idx = indices[mask]
        right_idx = indices[~mask]

        left = self._build_tree(X, grad, hess, left_idx, depth + 1)
        right = self._build_tree(X, grad, hess, right_idx, depth + 1)
        return Node(feature=feature, threshold=threshold, left=left, right=right)
    
    # Fit
    def fit(self, X, grad, hess):
        """Fit the model."""
        X = np.array(X, dtype=np.float32)
        y = np.array(y).ravel()

        indices = np.arange(len(grad))
        if self.subsample < 1.0:
            sample_size = int(self.subsample * len(indices))
            indices = np.random.choice(indices, sample_size, replace=False)

        self.root = self._build_tree(X, grad, hess, indices)
        return self

    def _predict_one(self, x, node):
        """Traverse tree for one sample."""
        if node.leaf_weight is not None:
            return node.leaf_weight

        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)
        
    def predict(self, X):
        """Predict the model for a test case."""
        X = np.array(X)
        return np.array([self._predict_one(x, self.root) for x in X])


## XGBoost

In [ ]:
# We train a no of trees on residuals with former tree 
# # and learn from them by learning rate 

from sklearn.metrics import roc_auc_score 

class XGBoost: 
    def __init__(self, n_estimators = 100, subsample=0.8, learning_rate=0.01, 
                 scale_pos_weight = 1.0, max_depth= 3, min_child_weight=2, 
                 colsample_bytree=0.8, reg_alpha=0.0, reg_gamma=0.0, 
                 reg_lambda=1.0, random_state=42): 
        self.n_estimators = n_estimators 
        self.subsample = subsample 
        self.learning_rate = learning_rate 
        self.scale_pos_weight = scale_pos_weight 
        self.max_depth = max_depth 
        self.min_child_weight = min_child_weight 
        self.colsample_bytree = colsample_bytree 
        self.reg_alpha = reg_alpha 
        self.reg_gamma = reg_gamma 
        self.reg_lambda = reg_lambda 
        self.random_state = random_state 

        self.trees = [] 
        self.base_score = None 
        np.random.seed(random_state) 
        
    def fit(self, X, y): 
        X = np.array(X) 
        y = np.array(y).ravel().astype(np.float64) 
        
        n_pos = np.sum(y == 1) 
        n_neg = len(y) - n_pos 
        
        # p_eff = (self.scale_pos_weight * n_pos) / (self.scale_pos_weight * n_pos + n_neg) 
        # self.base_score = np.log(p_eff / (1-p_eff)) if 0 < p_eff < 1 else 0.0 
        p = np.clip(np.mean(y), 1e-6, 1 - 1e-6)
        self.base_score = np.log(p / (1 - p))

        print(f"XGBoost from Scratch -> base_score (raw logit) = {self.base_score:.6f} ", 
              f"-> initial prob = {1/(1+np.exp(-self.base_score)):.6f} ", 
              f"(scale_pos_weight={self.scale_pos_weight})") 
        
        pred = np.full(len(y), self.base_score, dtype=np.float64) 
        
        for t in range(self.n_estimators): 
            # 1.Current probability 
            p = 1/(1+np.exp(-np.clip(pred, -35, 35))) 
            
            # 2. Gradient & Hessian (with scale_pos_weight applied) 
            grad = p - y 
            hess = p*(1-p) 
            grad[y==1] *= self.scale_pos_weight 
            hess[y==1] *= self.scale_pos_weight 
            
            # 3. Build one tree on (grad, hess) 
            tree = DecisionTreeXGBoostStyle( 
                max_depth=self.max_depth, 
                min_child_weight=self.min_child_weight, 
                colsample_bytree=self.colsample_bytree, 
                reg_alpha=self.reg_alpha, 
                reg_gamma=self.reg_gamma, 
                reg_lambda=self.reg_lambda, 
                subsample=self.subsample 
            ) 
            
            tree.fit(X=X, grad=grad, hess=hess) 
            
            # 4. Add tree output (scaled by learning_rate) 
            tree_output = tree.predict(X) # raw leaf weights 
            pred += self.learning_rate * tree_output 

            self.trees.append(tree) 

            current_p = 1 / (1 + np.exp(-pred)) 
            train_auc = roc_auc_score(y_true=y, y_score=current_p) 
            
            if (t+1) % 20 == 0 or t == 0: 
                print(f"Tree {t+1}/{self.n_estimators} build. Train auc :{train_auc:.4f}") 
    
        return self 

    def predict_raw(self, X): 
        """Return raw legit scores before sigmoid""" 
        X = np.array(X) 
        raw = np.full(X.shape[0], self.base_score, dtype=np.float64) 
        
        for tree in self.trees: 
            raw += self.learning_rate * tree.predict(X) 
        return raw 
    
    def predict(self, X, threshold=0.1): 
        raw = self.predict_raw(X) 
        return (raw >= threshold).astype(int) 
    
    def predict_proba(self, X): 
        """Probability of class 1""" 
        raw = self.predict_raw(X) 
        return 1 / (1 + np.exp(-raw))

## Model Training

### Model - 1 (Balanced, strong baseline)

In [ ]:
model_xgb_1 = XGBoost(
    n_estimators=300,
    learning_rate=0.08,
    scale_pos_weight=27.5,

    max_depth=4,
    min_child_weight=8.0,

    colsample_bytree=0.8,
    subsample=0.8,

    reg_alpha=1.0,
    reg_lambda=8.0,
    reg_gamma=0.2,

    random_state=42
)

import time
start = time.time()
model_xgb_1.fit(X_train, y_train)
end = time.time()
print("XG Boost Model 1 Training time: ", (end - start)/60, "mins.")

In [ ]:
y_preds_proba_1 = model_xgb_1.predict_proba(X_valid)

from sklearn.metrics import roc_auc_score

auc_1 = roc_auc_score(y_valid, y_preds_proba_1)
gini_1 = 2 * auc_1 - 1
print("AUC Score for Model - 1: ", auc_1)
print("Gini Score for Model - 1: ", gini_1)

### Model - 2 (Regularized / low-variance model)

In [ ]:
model_xgb_2 = XGBoost(
    n_estimators=300,
    learning_rate=0.06,
    scale_pos_weight=27.5,

    max_depth=3,
    min_child_weight=15.0,

    colsample_bytree=0.6,
    subsample=0.9,

    reg_alpha=3.0,
    reg_lambda=15.0,
    reg_gamma=0.5,

    random_state=42
)

import time
start = time.time()
model_xgb_2.fit(X_train, y_train)
end = time.time()
print("XG Boost Model 1 Training time: ", (end - start)/60, "mins.")

In [ ]:
y_preds_proba_2 = model_xgb_2.predict_proba(X_valid)

from sklearn.metrics import roc_auc_score

auc_2 = roc_auc_score(y_valid, y_preds_proba_2)
gini_2 = 2 * auc_2 - 1
print("AUC Score for Model - 1: ", auc_2)
print("Gini Score for Model - 1: ", gini_2)

### Model - 3 (High variance learner)

In [ ]:
model_xgb_3 = XGBoost(
    n_estimators=200,
    learning_rate=0.1,
    scale_pos_weight=27.5,

    max_depth=5,
    min_child_weight=6.0,

    colsample_bytree=0.6,
    subsample=0.7,

    reg_alpha=0.5,
    reg_lambda=5.0,
    reg_gamma=0.1,

    random_state=42
)

import time
start = time.time()
model_xgb_3.fit(X_train, y_train)
end = time.time()
print("XG Boost Model 1 Training time: ", (end - start)/60, "mins.")

In [ ]:
y_preds_proba_3 = model_xgb_3.predict_proba(X_valid)

from sklearn.metrics import roc_auc_score

auc_3 = roc_auc_score(y_valid, y_preds_proba_3)
gini_3 = 2 * auc_3 - 1
print("AUC Score for Model - 1: ", auc_3)
print("Gini Score for Model - 1: ", gini_3)